# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begs44/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**The contract in five plain answers (Refresh lane):**

1. **One row means:** one content item (page) in March 2026, summarized from its daily rows. The raw table's row is one content item × one day.
2. **Table(s):** `fact_content_daily_performance` (partition `month=2026-03`) for search metrics, joined to `dim_content` on `content_hash_id` for the creation date.
3. **Time window:** March 2026, a mid-panel month (June 2026 stays sealed). Decision moment = **16 March**: features use **1–15 March**, the label uses **16–31 March**.
4. **What I predict (proxy):** `declined` = 1 if a page's average daily impressions in 16–31 March are more than 20% below its average in 1–15 March. A proxy for "worth a refresh review", not lost revenue.
5. **One thing I deliberately exclude:** every number dated 16–31 March as a feature, because the label is computed from it. Using it would be leakage.

In [8]:
%pip -q install duckdb
import os
import duckdb
import pandas as pd
import numpy as np

# Token from the Colab Secret named HF_TOKEN (key panel on the left) - never pasted into a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"                                    # a mid-panel month; June 2026 stays sealed
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
DECISION = "2026-03-16"                              # decision moment: features before it, label after it

schema = con.sql(f"DESCRIBE SELECT * FROM {FACT}").df()[["column_name", "column_type"]]
FLAGS = [c for c in schema["column_name"] if c.endswith("data_available")]
GSC_FLAG = next((c for c in FLAGS if c.startswith("gsc")), FLAGS[0] if FLAGS else None)
assert GSC_FLAG, "no *_data_available flag found - check the column list below"
print(f"Table: fact_content_daily_performance, partition month={MONTH} | {len(schema)} columns")
print(f"Availability flags found: {FLAGS} -> using {GSC_FLAG} for the search metrics")
print(schema.to_string(index=False))

Table: fact_content_daily_performance, partition month=2026-03 | 31 columns
Availability flags found: ['gsc_data_available', 'ga4_data_available'] -> using gsc_data_available for the search metrics
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Features (1–15 March only):** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, days with impressions (from `report_date`), page age (from `dim_content.content_created_date`).
* **Label source (never a feature):** `gsc_impressions` in 16–31 March.
* **Context (group / join only):** `client_hash_id`, `content_hash_id` (pseudonyms), and the availability flag (used to filter rows, not to learn from).
* **Excluded, with why:**
  - Everything dated 16–31 March as a feature: it is the label window.
  - `is_published` / `is_deleted`: status at export time (July 2026), after the decision moment, so future information.
  - GA4 columns: not needed for a search-impressions label, and zero-filled before each client's GA4 start.

In [9]:
CONTRACT = {
    "label source (never a feature)": ["gsc_impressions after the decision moment (16-31 Mar)"],
    "features (1-15 Mar only)": ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "report_date", "content_created_date"],
    "context (group / join only)": ["client_hash_id", "content_hash_id", GSC_FLAG],
    "excluded": ["everything dated 16-31 Mar as a feature", "is_published / is_deleted (status at export, July 2026)",
                 "GA4 columns (not needed; zero-filled before each client's GA4 start)"],
}
fact_cols = set(schema["column_name"])
dim_cols = set(con.sql(f"DESCRIBE SELECT * FROM {DIM}").df()["column_name"])
needed = {"report_date": fact_cols, "client_hash_id": fact_cols, "content_hash_id": fact_cols,
          "gsc_impressions": fact_cols, "gsc_clicks": fact_cols, "gsc_avg_position": fact_cols,
          GSC_FLAG: fact_cols, "content_created_date": dim_cols}
missing = [c for c, cols in needed.items() if c not in cols]
assert not missing, f"columns not found: {missing}"
for bucket, items in CONTRACT.items():
    print(f"{bucket:<32} {items}")
print("\nAll contract columns exist in the release (fact + dim_content).")

label source (never a feature)   ['gsc_impressions after the decision moment (16-31 Mar)']
features (1-15 Mar only)         ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'report_date', 'content_created_date']
context (group / join only)      ['client_hash_id', 'content_hash_id', 'gsc_data_available']
excluded                         ['everything dated 16-31 Mar as a feature', 'is_published / is_deleted (status at export, July 2026)', "GA4 columns (not needed; zero-filled before each client's GA4 start)"]

All contract columns exist in the release (fact + dim_content).


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Three verification queries** (code cell A), all on the March 2026 partition:
1. **Grain:** no duplicate (report_date, client, content) keys → one row really is one content item on one day.
2. **Count + span:** rows, first/last day, number of content items and clients in my slice.
3. **Availability:** the flag can be TRUE, FALSE or NULL across the release, so I filter with `IS TRUE` and show how many rows survive (36.7% in March). This month has no NULL flags, so a naive `NOT flag` filter gives the same count here, but `IS TRUE` stays correct in months and clients where NULLs do appear.

**Five features** (code cell B), built in DuckDB from the same month, each with a "knowable at the decision moment because…" line. Only pages with ≥ 50 impressions in 1–15 March are kept, so a change can be judged.

**The trap** (code cell C): I add ONE label-derived column on purpose (impressions *after* 16 March, which the label is computed from), watch the grouped-by-client ROC-AUC jump toward 1.0, then delete it and keep the honest number.

In [10]:
# ---- Query 1: grain. One row should be one content item on one day. Zero duplicate keys = grain holds.
dups = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Query 1 (grain): duplicate (report_date, client, content) keys = {len(dups)} -> "
      f"{'grain holds: 1 row = 1 content item x 1 day' if dups.empty else 'GRAIN BROKEN'}")

# ---- Query 2: row count and date span of my slice
q2 = con.sql(f"""
    SELECT COUNT(*) AS rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT report_date) AS days, COUNT(DISTINCT content_hash_id) AS content_items,
           COUNT(DISTINCT client_hash_id) AS clients
    FROM {FACT}
""").df()
print("\nQuery 2 (count + span):\n" + q2.to_string(index=False))

# ---- Query 3: availability. The flag is three-valued (TRUE / FALSE / NULL), so filter with IS TRUE.
q3 = con.sql(f"""
    SELECT COUNT(*)                                          AS all_rows,
           COUNT(*) FILTER (WHERE {GSC_FLAG} IS TRUE)       AS kept_is_true,
           COUNT(*) FILTER (WHERE {GSC_FLAG} IS FALSE)      AS flag_false,
           COUNT(*) FILTER (WHERE {GSC_FLAG} IS NULL)       AS flag_null,
           COUNT(*) FILTER (WHERE NOT {GSC_FLAG})           AS naive_not_flag
    FROM {FACT}
""").df()
print("\nQuery 3 (availability):\n" + q3.to_string(index=False))
r = q3.iloc[0]
if r.flag_null == 0:
    note = ("No NULL flags in this month, so 'NOT flag' happens to give the same count here - "
            "IS TRUE is still the safe filter, because other months and clients do carry NULL flags.")
else:
    note = (f"A naive 'NOT flag' filter would catch only {r.naive_not_flag:,} of the "
            f"{r.flag_false + r.flag_null:,} unavailable rows - NULLs slip through.")
print(f"-> {r.kept_is_true:,} of {r.all_rows:,} rows survive '{GSC_FLAG} IS TRUE' ({r.kept_is_true / r.all_rows:.1%}). {note}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 (grain): duplicate (report_date, client, content) keys = 0 -> grain holds: 1 row = 1 content item x 1 day


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 2 (count + span):
   rows  first_day   last_day  days  content_items  clients
9841378 2026-03-01 2026-03-31    31         331437       55


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 3 (availability):
 all_rows  kept_is_true  flag_false  flag_null  naive_not_flag
  9841378       3611061     6230317          0         6230317
-> 3,611,061 of 9,841,378 rows survive 'gsc_data_available IS TRUE' (36.7%). No NULL flags in this month, so 'NOT flag' happens to give the same count here - IS TRUE is still the safe filter, because other months and clients do carry NULL flags.


In [11]:
# ---- Five features, built inside DuckDB from the SAME month. Features use 1-15 March only;
# the label uses 16-31 March only, so nothing a feature sees comes after the decision moment.
frame = con.sql(f"""
    WITH f AS (
        SELECT client_hash_id, content_hash_id, CAST(report_date AS DATE) AS d,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM {FACT}
        WHERE {GSC_FLAG} IS TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN d <  DATE '{DECISION}' THEN gsc_impressions ELSE 0 END)                    AS imp_before,
               SUM(CASE WHEN d <  DATE '{DECISION}' THEN gsc_clicks ELSE 0 END)                         AS clicks_before,
               AVG(CASE WHEN d <  DATE '{DECISION}' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_before,
               COUNT(DISTINCT CASE WHEN d < DATE '{DECISION}' AND gsc_impressions > 0 THEN d END)       AS days_seen_before,
               SUM(CASE WHEN d >= DATE '{DECISION}' THEN gsc_impressions ELSE 0 END)                    AS imp_after
        FROM f
        GROUP BY 1, 2
    )
    SELECT a.*,
           date_diff('day', TRY_CAST(c.content_created_date AS DATE), DATE '{DECISION}') AS age_days
    FROM agg a
    LEFT JOIN (SELECT content_hash_id, ANY_VALUE(content_created_date) AS content_created_date
               FROM {DIM} GROUP BY 1) c USING (content_hash_id)
    WHERE a.imp_before >= 50                                   -- enough history to judge a change
""").df()

n0 = len(frame)
frame = frame[~(frame["age_days"] < 0)].copy()                  # registered after the decision moment
# Label (proxy for "worth a refresh review"): average daily impressions fell >20% after the decision moment
frame["declined"] = ((frame["imp_after"] / 16) < 0.8 * (frame["imp_before"] / 15)).astype(int)
frame["log_imp_before"] = np.log1p(frame["imp_before"])
frame["log_clicks_before"] = np.log1p(frame["clicks_before"])

FEATURES = {
    "log_imp_before":    "knowable at the decision moment because it sums impressions from 1-15 March only",
    "log_clicks_before": "knowable at the decision moment because it sums clicks from 1-15 March only",
    "position_before":   "knowable at the decision moment because it averages positions from 1-15 March only",
    "days_seen_before":  "knowable at the decision moment because it counts days with impressions in 1-15 March",
    "age_days":          "knowable at the decision moment because the creation date is fixed before 16 March",
}
print(f"Feature frame: {len(frame):,} content items (>= 50 impressions in 1-15 Mar; {n0 - len(frame)} created later dropped) "
      f"from {frame['client_hash_id'].nunique()} clients | label base rate {frame['declined'].mean():.3f}\n")
for f_, why in FEATURES.items():
    print(f"- {f_:<18} {why}")
print()
print(frame[list(FEATURES) + ["declined"]].describe().T[["count", "mean", "50%", "min", "max"]].round(2).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 92,548 content items (>= 50 impressions in 1-15 Mar; 0 created later dropped) from 40 clients | label base rate 0.327

- log_imp_before     knowable at the decision moment because it sums impressions from 1-15 March only
- log_clicks_before  knowable at the decision moment because it sums clicks from 1-15 March only
- position_before    knowable at the decision moment because it averages positions from 1-15 March only
- days_seen_before   knowable at the decision moment because it counts days with impressions in 1-15 March
- age_days           knowable at the decision moment because the creation date is fixed before 16 March

                     count    mean     50%   min     max
log_imp_before     92548.0    6.14    6.01  3.93   11.99
log_clicks_before  92548.0    0.84    0.69  0.00    7.78
position_before    92545.0   14.02    8.24  0.04  127.62
days_seen_before   92548.0   14.05   15.00  1.00   15.00
age_days           92548.0  183.38  178.00  2.00  479.00
declined 

In [12]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

def quick_score(cols):
    """ROC-AUC, mean of 5 folds grouped by client: every client is scored by a model that never saw it."""
    X, y, g = frame[cols], frame["declined"], frame["client_hash_id"]
    scores = []
    for tr, te in GroupKFold(n_splits=5).split(X, y, g):
        m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
        m.fit(X.iloc[tr], y.iloc[tr])
        scores.append(roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te])[:, 1]))
    return np.mean(scores)

honest_cols = list(FEATURES)
honest = quick_score(honest_cols)
print(f"Base rate {frame['declined'].mean():.3f} | random ranking = ROC-AUC 0.50")
print(f"Honest 5 features:                         ROC-AUC = {honest:.3f}")

# THE TRAP: add ONE label-derived column on purpose. The label is computed from imp_after.
frame["LEAK_log_imp_after"] = np.log1p(frame["imp_after"])
leaky = quick_score(honest_cols + ["LEAK_log_imp_after"])
print(f"+ impressions AFTER the decision (leak):   ROC-AUC = {leaky:.3f}   <- jumps toward perfect")

# Delete it and keep the honest number
frame = frame.drop(columns=["LEAK_log_imp_after"])
assert "LEAK_log_imp_after" not in frame.columns
print(f"Leak deleted. The number I keep:           ROC-AUC = {quick_score(honest_cols):.3f}")

Base rate 0.327 | random ranking = ROC-AUC 0.50
Honest 5 features:                         ROC-AUC = 0.615
+ impressions AFTER the decision (leak):   ROC-AUC = 1.000   <- jumps toward perfect
Leak deleted. The number I keep:           ROC-AUC = 0.615


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: two short half-month windows inside one month, on an unbalanced panel.** A 15-day feature window and a 16-day label window are noisy: they hold different numbers of weekend days, a single news spike or holiday can swing a page by 20%, and pages with only 50–99 impressions cross the threshold easily by chance. Clients also have very different history depth and page counts, so one large client can dominate. The label is a short-term proxy, not proof that a refresh is needed, and nothing here is causal.

In [13]:
# Limitation check: two short half-month windows in one month, on an unbalanced panel.
days = pd.date_range(f"{MONTH}-01", f"{MONTH}-31")
before, after = days[days < DECISION], days[days >= DECISION]
print(f"Feature window: {len(before)} days ({(before.dayofweek >= 5).sum()} weekend days) | "
      f"label window: {len(after)} days ({(after.dayofweek >= 5).sum()} weekend days)")
share = frame["client_hash_id"].value_counts(normalize=True)
print(f"Clients in the frame: {len(share)} | largest client holds {share.iloc[0]:.0%} of the content items")
print(f"Content items with only 50-99 impressions in 1-15 Mar: {(frame['imp_before'] < 100).mean():.0%} "
      "(small counts make a >20% swing easy by chance)")

Feature window: 15 days (5 weekend days) | label window: 16 days (4 weekend days)
Clients in the frame: 40 | largest client holds 22% of the content items
Content items with only 50-99 impressions in 1-15 Mar: 16% (small counts make a >20% swing easy by chance)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.